<a href="https://colab.research.google.com/github/Kamalll7/Final_Project_PCD-Kelompok_6/blob/main/notebook/01_Helmet_Detection_CHT_HOG_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## 1. Pendahuluan

### 1.1 Latar Belakang Singkat

Sepeda motor adalah kendaraan yang paling banyak digunakan di Indonesia. Undang-Undang
Nomor 22 Tahun 2009 mewajibkan pengendara sepeda motor memakai helm yang memenuhi standar.
Pengawasan manual terhadap kepatuhan ini tidak efisien pada skala lalu lintas nyata, sehingga
diperlukan sistem deteksi otomatis berbasis pengolahan citra digital.

### 1.2 Rumusan Masalah yang Dijawab Notebook Ini

> **Bagaimana mendeteksi keberadaan helm pada pengendara sepeda motor menggunakan metode
> pengolahan citra digital dan machine learning?** (Rumusan Masalah #1)

### 1.3 Tujuan

Mengembangkan dan mengevaluasi metode deteksi keberadaan helm pada pengendara sepeda motor
menggunakan **CHT** untuk melokalisasi area kepala, **HOG** untuk mengekstraksi deskriptor
tekstur/gradien, dan **SVM** untuk klasifikasi biner.

### 1.4 Mengapa CHT + HOG + SVM?

Berdasarkan kajian pustaka pada proposal (§2.3–§2.5):

- **CHT** efektif melokalisasi objek berbentuk lingkaran. Helm dan kepala manusia sama-sama
  menyerupai lingkaran, sehingga CHT dapat mempersempit area pencarian.
- **CHT saja tidak cukup** — helm dan kepala tanpa helm sama-sama berbentuk lingkaran. Informasi
  geometris semata tidak dapat membedakan keduanya (temuan e Silva et al., 2017).
- **HOG** menangkap perbedaan karakteristik permukaan: helm umumnya halus dan mengilap,
  sedangkan rambut/kepala memiliki tekstur yang lebih kompleks.
- **SVM kernel linear** memberikan keseimbangan terbaik antara akurasi dan kecepatan komputasi
  pada fitur HOG berdimensi tinggi (Hsu et al., 2015).

Kombinasi CHT (lokalisasi) + HOG (deskripsi) inilah yang disebut **deskriptor hybrid** pada proposal.

---
## 2. Setup Environment

Notebook ini dirancang untuk berjalan di **Google Colab** (sesuai §3.10 proposal) maupun secara lokal.

**Perangkat lunak yang digunakan** (§3.10):
`Python`, `OpenCV` (pengolahan citra & CHT), `scikit-image` (HOG), `scikit-learn` (SVM & metrik),
`NumPy`, `Matplotlib`, `Pandas`.

In [ ]:
import sys, subprocess

REQUIRED = ["opencv-python-headless", "scikit-image", "scikit-learn",
            "matplotlib", "pandas", "numpy", "pillow"]

def _ensure(pkgs):
    missing = []
    for p in pkgs:
        mod = {"opencv-python-headless": "cv2", "scikit-image": "skimage",
               "scikit-learn": "sklearn", "pillow": "PIL"}.get(p, p)
        try:
            __import__(mod)
        except ImportError:
            missing.append(p)
    if missing:
        print("Memasang:", missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)

_ensure(REQUIRED)

In [ ]:
import os, json, random, warnings, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, cohen_kappa_score,
                             roc_curve, auc, classification_report)

warnings.filterwarnings("ignore")

# ---- Reproducibility -------------------------------------------------------
SEED = 7
random.seed(SEED)
np.random.seed(SEED)

# ---- Tampilan plot ---------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 110,
    "font.size": 9,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
})

print("NumPy       :", np.__version__)
print("OpenCV      :", cv2.__version__)
import sklearn, skimage
print("scikit-learn:", sklearn.__version__)
print("scikit-image:", skimage.__version__)

---
## 3. Akuisisi Dataset

Dataset: **Rider, With Helmet, Without Helmet, Number Plate** (Kaggle, `aneesarom`).

Struktur dataset mengikuti format anotasi **YOLO**:

```
dataset/
├── train/
│   ├── images/   *.jpg
│   └── labels/   *.txt      # satu baris per objek: <class> <x_center> <y_center> <w> <h>
└── val/
    ├── images/
    └── labels/
```

Nilai koordinat pada label YOLO **dinormalisasi** terhadap lebar dan tinggi citra (rentang 0–1),
sehingga perlu dikonversi ke koordinat piksel sebelum digunakan.

**Kelas** (`classes.txt`):

| ID | Nama kelas |
|---|---|
| 0 | `with helmet` |
| 1 | `without helmet` |
| 2 | `rider` |
| 3 | `number plate` |

> **Catatan penting.** Dataset ini menyediakan anotasi `with helmet` / `without helmet` yang
> sudah terasosiasi dengan setiap pengendara. Anotasi inilah yang menjadi **ground truth**
> untuk melatih dan mengevaluasi classifier SVM pada jalur helm.

In [ ]:
DATASET_SLUG = "aneesarom/rider-with-helmet-without-helmet-number-plate"
CLASS_NAMES  = ["with helmet", "without helmet", "rider", "number plate"]

def _looks_like_dataset(p: Path) -> bool:
    return (p / "train" / "images").is_dir() and (p / "train" / "labels").is_dir()

def resolve_dataset() -> Path:
    """Cari folder dataset: env var -> path lokal umum -> cache kagglehub -> unduh."""
    candidates = []
    env = os.environ.get("PCD_DATASET_DIR")
    if env:
        candidates.append(Path(env))
    candidates += [Path("./dataset"), Path("../dataset"), Path("/content/dataset"),
                   Path.home() / ".cache/kagglehub/datasets/aneesarom/rider-with-helmet-without-helmet-number-plate"]

    for base in candidates:
        if not base.exists():
            continue
        if _looks_like_dataset(base):
            return base
        # kagglehub menyimpan pada subfolder bernomor versi
        for sub in base.rglob("train"):
            if (sub / "images").is_dir() and (sub / "labels").is_dir():
                return sub.parent

    print("Dataset belum ditemukan secara lokal, mengunduh via kagglehub ...")
    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=False)
        import kagglehub
    root = Path(kagglehub.dataset_download(DATASET_SLUG))
    for sub in [root] + list(root.rglob("*")):
        if _looks_like_dataset(sub):
            return sub
    raise FileNotFoundError("Struktur dataset tidak dikenali setelah unduhan.")

DATA_ROOT = resolve_dataset()
print("Dataset root :", DATA_ROOT.resolve())
print("Train images :", len(list((DATA_ROOT / 'train' / 'images').glob('*'))))
print("Val images   :", len(list((DATA_ROOT / 'val' / 'images').glob('*'))))

### 3.1 Pembaca Citra yang Robust

Satu berkas pada dataset berekstensi `.jpg` namun sebenarnya berformat **AVIF**, sehingga
`cv2.imread()` mengembalikan `None`. Karena itu digunakan pembaca dengan fallback ke `Pillow`.
Penanganan kasus seperti ini penting agar pipeline tidak diam-diam kehilangan data.

In [ ]:
def imread_robust(path) -> np.ndarray:
    """Baca citra sebagai BGR; fallback ke Pillow untuk format tak lazim (mis. AVIF)."""
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is not None:
        return img
    from PIL import Image
    with Image.open(path) as im:
        arr = np.array(im.convert("RGB"))
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)


def parse_label_file(path: Path):
    """Baca label YOLO -> list dict dengan koordinat ternormalisasi (x1,y1,x2,y2)."""
    boxes = []
    for line in path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls = int(parts[0])
        x, y, w, h = map(float, parts[1:5])
        boxes.append({"cls": cls,
                      "x1": x - w / 2, "y1": y - h / 2,
                      "x2": x + w / 2, "y2": y + h / 2})
    return boxes


def build_index(root: Path) -> pd.DataFrame:
    """Bangun tabel: satu baris = satu citra, beserta path & jumlah objek per kelas."""
    rows = []
    for split in ("train", "val"):
        for lbl in sorted((root / split / "labels").glob("*.txt")):
            stem = lbl.stem
            img_path = root / split / "images" / (stem + ".jpg")
            if not img_path.exists():
                # cari ekstensi lain
                hits = list((root / split / "images").glob(stem + ".*"))
                if not hits:
                    continue
                img_path = hits[0]
            boxes = parse_label_file(lbl)
            row = {"split": split, "stem": stem, "image_path": str(img_path),
                   "n_objects": len(boxes), "boxes": boxes}
            for cid, cname in enumerate(CLASS_NAMES):
                row["n_" + cname] = sum(1 for b in boxes if b["cls"] == cid)
            rows.append(row)
    return pd.DataFrame(rows)

df = build_index(DATA_ROOT)
print(f"Total citra terindeks: {len(df)}")
df.head(3)[["split", "stem", "n_objects", "n_with helmet", "n_without helmet", "n_rider", "n_number plate"]]

Total citra terindeks: 124


,split,stem,n_objects,n_with helmet,n_without helmet,n_rider,n_number plate
0,train,new1,3,1,0,1,1
1,train,new10,4,0,2,1,1
2,train,new100,4,0,2,1,1


In [ ]:
# Verifikasi bahwa seluruh citra dapat dibaca
bad = []
for p in df["image_path"]:
    if imread_robust(p) is None:
        bad.append(p)
print("Citra gagal dibaca:", len(bad), bad)

total = {c: int(df["n_" + c].sum()) for c in CLASS_NAMES}
print("\nJumlah bounding box per kelas:")
for k, v in total.items():
    print(f"  {k:16s} : {v}")
print(f"  {'TOTAL':16s} : {sum(total.values())}")

---
## 4. Eksplorasi Data (EDA)

Tahap ini bertujuan memahami karakteristik dataset sebelum merancang pipeline, khususnya:

1. **Distribusi kelas** — apakah data seimbang?
2. **Ukuran & rasio objek** — menentukan ukuran *window* HOG.
3. **Posisi spasial helm relatif terhadap pengendara** — dasar penentuan RoI.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

# --- distribusi bounding box per kelas
counts = [total[c] for c in CLASS_NAMES]
colors = ["#2a9d8f", "#e76f51", "#457b9d", "#e9c46a"]
axes[0].bar(CLASS_NAMES, counts, color=colors, edgecolor="black", linewidth=0.6)
for i, v in enumerate(counts):
    axes[0].text(i, v + 2, str(v), ha="center", fontsize=9, fontweight="bold")
axes[0].set_title("Jumlah bounding box per kelas")
axes[0].set_ylabel("Jumlah objek")
axes[0].tick_params(axis="x", rotation=15)

# --- jumlah citra yang memuat tiap kelas
img_counts = [int((df["n_" + c] > 0).sum()) for c in CLASS_NAMES]
axes[1].bar(CLASS_NAMES, img_counts, color=colors, edgecolor="black", linewidth=0.6)
for i, v in enumerate(img_counts):
    axes[1].text(i, v + 1, str(v), ha="center", fontsize=9, fontweight="bold")
axes[1].set_title("Jumlah citra yang memuat kelas")
axes[1].set_ylabel("Jumlah citra")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

print("Rata-rata pengendara per citra : %.2f" % (total["rider"] / len(df)))
print("Citra tanpa anotasi pengendara: %d" % int((df["n_rider"] == 0).sum()))

### 4.1 Posisi Spasial Helm terhadap Pengendara

Analisis ini menjadi **dasar kuantitatif penentuan Region of Interest (RoI)** pada §3.5 proposal.
Dengan mengetahui di mana helm biasanya berada relatif terhadap *bounding box* pengendara,
kita dapat mempersempit area pencarian secara signifikan.

In [ ]:
def iou_norm(a, b):
    """IoU dua box ternormalisasi (x1,y1,x2,y2)."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def box_tuple(b):
    return (b["x1"], b["y1"], b["x2"], b["y2"])

# Kumpulkan relasi helm <-> pengendara
rel = []
for _, row in df.iterrows():
    riders = [b for b in row["boxes"] if b["cls"] == 2]
    helmets = [b for b in row["boxes"] if b["cls"] in (0, 1)]
    for r in riders:
        rb = box_tuple(r)
        rw, rh = rb[2] - rb[0], rb[3] - rb[1]
        for h in helmets:
            hb = box_tuple(h)
            if iou_norm(rb, hb) <= 0.001:
                continue
            rel.append({
                "cx": ((hb[0] + hb[2]) / 2 - rb[0]) / rw,
                "cy": ((hb[1] + hb[3]) / 2 - rb[1]) / rh,
                "w_ratio": (hb[2] - hb[0]) / rw,
                "h_ratio": (hb[3] - hb[1]) / rh,
                "label": CLASS_NAMES[h["cls"]],
            })

rel_df = pd.DataFrame(rel)
print("Pasangan helm-pengendara:", len(rel_df))
print(rel_df.groupby("label")[["cy", "w_ratio", "h_ratio"]].agg(["mean", "median", "count"]).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

for lab, sub in rel_df.groupby("label"):
    axes[0].hist(sub["cy"], bins=25, alpha=0.65, label=lab)
axes[0].axvline(0.30, color="crimson", ls="--", lw=1.4)
axes[0].text(0.31, axes[0].get_ylim()[1]*0.9, "batas RoI\n(30% tinggi pengendara)",
             color="crimson", fontsize=8, va="top")
axes[0].set_xlabel("Posisi vertikal pusat helm / tinggi pengendara")
axes[0].set_ylabel("Frekuensi")
axes[0].set_title("Distribusi posisi vertikal helm")
axes[0].legend(fontsize=8)

for lab, sub in rel_df.groupby("label"):
    axes[1].scatter(sub["w_ratio"], sub["h_ratio"], s=14, alpha=0.6, label=lab)
axes[1].set_xlabel("Lebar helm / lebar pengendara")
axes[1].set_ylabel("Tinggi helm / tinggi pengendara")
axes[1].set_title("Ukuran relatif helm")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("Proporsi helm yang pusatnya berada di 30%% atas pengendara: %.1f%%"
      % (100 * (rel_df["cy"] <= 0.30).mean()))

In [ ]:
# Visualisasi contoh citra beserta anotasinya
COLORS = {0: "#2a9d8f", 1: "#e76f51", 2: "#457b9d", 3: "#e9c46a"}

sample = df.sample(6, random_state=SEED)
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    img = imread_robust(row["image_path"])
    H, W = img.shape[:2]
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for b in row["boxes"]:
        x1, y1 = b["x1"] * W, b["y1"] * H
        w, h = (b["x2"] - b["x1"]) * W, (b["y2"] - b["y1"]) * H
        ax.add_patch(Rectangle((x1, y1), w, h, fill=False,
                               edgecolor=COLORS[b["cls"]], lw=1.6))
        ax.text(x1, max(y1 - 3, 8), CLASS_NAMES[b["cls"]], fontsize=6,
                color="white", bbox=dict(fc=COLORS[b["cls"]], ec="none", pad=1.2))
    ax.set_title(f"{row['stem']}  ({W}×{H})", fontsize=8)
    ax.axis("off")

handles = [Rectangle((0, 0), 1, 1, fc=COLORS[i], ec="none") for i in range(4)]
fig.legend(handles, CLASS_NAMES, loc="lower center", ncol=4, frameon=False, fontsize=9)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

---
## 5. Pra-pemrosesan Citra  ·  *(Proposal §3.4)*

Sesuai proposal, pra-pemrosesan terdiri atas tiga tahap:

| Tahap | Operasi | Alasan |
|---|---|---|
| 1 | **Resize** ke ukuran seragam | mengurangi jumlah piksel → beban komputasi turun, informasi relevan dipertahankan |
| 2 | **Konversi Grayscale** | fokus pada bentuk/tepi/intensitas; HOG memang bekerja pada variasi intensitas, bukan warna |
| 3 | **Peningkatan kontras (CLAHE)** | mempertegas perbedaan intensitas objek vs latar belakang |

> **Catatan metodologis.** Pada tahap ini diterapkan **CLAHE** (*Contrast Limited Adaptive
> Histogram Equalization*) sebagai metode peningkatan kontras. CLAHE dipilih dibanding
> ekualisasi histogram global karena bersifat **adaptif per wilayah** dan **membatasi
> amplifikasi noise** (*clip limit*), sehingga cocok untuk citra lalu lintas dengan
> pencahayaan tidak merata.

In [ ]:
IMG_SIZE = (128, 128)      # (lebar, tinggi) untuk tampilan/analisis
CLAHE_CLIP = 2.0
CLAHE_GRID = (8, 8)

_clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)

def to_grayscale(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def resize_img(img, size=IMG_SIZE):
    return cv2.resize(img, size, interpolation=cv2.INTER_AREA)

def enhance_contrast(gray):
    return _clahe.apply(gray)

def preprocess(img, size=IMG_SIZE):
    """Pipeline pra-pemrosesan: resize -> grayscale -> CLAHE."""
    small = resize_img(img, size)
    gray = to_grayscale(small)
    return enhance_contrast(gray)

print("Fungsi pra-pemrosesan siap.")

In [ ]:
# Demonstrasi tahapan pra-pemrosesan pada satu citra
demo = df[df["n_rider"] > 0].iloc[0]
img = imread_robust(demo["image_path"])
small = resize_img(img)
gray = to_grayscale(small)
eq = enhance_contrast(gray)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"1. Citra asli\n{img.shape[1]}×{img.shape[0]}")
axes[1].imshow(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"2. Resize\n{IMG_SIZE[0]}×{IMG_SIZE[1]}")
axes[2].imshow(gray, cmap="gray")
axes[2].set_title("3. Grayscale")
axes[3].imshow(eq, cmap="gray")
axes[3].set_title("4. CLAHE")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

# Perbandingan histogram sebelum/sesudah CLAHE
fig, axes = plt.subplots(1, 2, figsize=(10, 2.8))
axes[0].hist(gray.ravel(), bins=64, color="#457b9d")
axes[0].set_title("Histogram grayscale")
axes[1].hist(eq.ravel(), bins=64, color="#2a9d8f")
axes[1].set_title("Histogram setelah CLAHE")
for ax in axes:
    ax.set_xlabel("Intensitas piksel")
    ax.set_ylabel("Frekuensi")
plt.tight_layout()
plt.show()

---
## 6. Penentuan Region of Interest (RoI)  ·  *(Proposal §3.5)*

Citra pada dataset masih memuat area yang tidak relevan (jalan, latar belakang, bodi motor).
RoI memfokuskan proses pada area yang mengandung objek yang diteliti sehingga:

- fitur yang diekstraksi lebih merepresentasikan objek,
- jumlah data yang diproses berkurang → pengolahan lebih efisien.

### Dua tingkat RoI pada jalur helm

1. **RoI pengendara (rider)** — diperoleh dari anotasi kelas `rider`. Pada sistem yang berjalan
   penuh, tahap ini setara dengan deteksi kendaraan (§2.2 proposal) yang mempersempit pencarian
   sebelum atribut dideteksi — pendekatan yang terbukti mengurangi *false positive* dan
   mempercepat deteksi (Hsu et al., 2015).
2. **RoI kepala** — diambil dari **30% bagian atas** *bounding box* pengendara, berdasarkan
   hasil EDA di §4.1 yang menunjukkan mayoritas pusat helm berada pada rentang tersebut.

Di dalam RoI kepala inilah **CHT** dijalankan.

In [ ]:
HEAD_BAND_TOP    = 0.00    # awal pita kepala (relatif tinggi pengendara)
HEAD_BAND_BOTTOM = 0.30    # akhir pita kepala

def rider_roi_px(img_shape, rider_box, top=HEAD_BAND_TOP, bottom=HEAD_BAND_BOTTOM,
                 x_lo=0.00, x_hi=1.00):
    """Konversi box pengendara ternormalisasi -> RoI kepala dalam koordinat piksel."""
    H, W = img_shape[:2]
    x1, y1, x2, y2 = rider_box
    w, h = x2 - x1, y2 - y1
    X1 = int(np.clip((x1 + x_lo * w) * W, 0, W - 1))
    X2 = int(np.clip((x1 + x_hi * w) * W, 0, W - 1))
    Y1 = int(np.clip((y1 + top * h) * H, 0, H - 1))
    Y2 = int(np.clip((y1 + bottom * h) * H, 0, H - 1))
    return X1, Y1, X2, Y2


def crop_px(img, box):
    X1, Y1, X2, Y2 = box
    if X2 - X1 < 8 or Y2 - Y1 < 8:
        return None
    return img[Y1:Y2, X1:X2]

# Demonstrasi RoI
row = df[df["n_rider"] > 0].iloc[1]
img = imread_robust(row["image_path"])
H, W = img.shape[:2]
rider = [b for b in row["boxes"] if b["cls"] == 2][0]
roi_box = rider_roi_px(img.shape, box_tuple(rider))
roi_img = crop_px(img, roi_box)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
x1, y1 = rider["x1"]*W, rider["y1"]*H
axes[0].add_patch(Rectangle((x1, y1), (rider["x2"]-rider["x1"])*W,
                            (rider["y2"]-rider["y1"])*H, fill=False, ec="#457b9d", lw=2))
axes[0].set_title("Citra + box pengendara")

axes[1].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[1].add_patch(Rectangle((roi_box[0], roi_box[1]), roi_box[2]-roi_box[0],
                            roi_box[3]-roi_box[1], fill=False, ec="crimson", lw=2))
axes[1].set_title("RoI kepala (30% atas)")

axes[2].imshow(cv2.cvtColor(roi_img, cv2.COLOR_BGR2RGB))
axes[2].set_title(f"Crop RoI  {roi_img.shape[1]}×{roi_img.shape[0]}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

---
## 7. Deteksi Kandidat Helm dengan Circular Hough Transform  ·  *(Proposal §2.3)*

### 7.1 Prinsip Kerja CHT

CHT mencari lingkaran pada citra dengan cara *voting* di ruang parameter $(a, b, r)$:

$$(x - a)^2 + (y - b)^2 = r^2$$

Setiap piksel tepi (*edge*) memberikan suara (*vote*) pada semua kemungkinan pusat lingkaran
$(a,b)$ untuk radius $r$ tertentu. Puncak akumulator dengan suara terbanyak menandakan
keberadaan lingkaran.

### 7.2 Parameter CHT yang Digunakan

| Parameter | Nilai | Peran |
|---|---|---|
| `dp` | 1.2 | resolusi akumulator (1 = sama dengan citra, >1 = lebih kasar & cepat) |
| `minDist` | ½ tinggi RoI | mencegah deteksi lingkaran ganda pada kepala yang sama |
| `param1` | 100 | ambang atas untuk detektor tepi Canny |
| `param2` | 18 | ambang akumulator — **semakin kecil, semakin banyak lingkaran terdeteksi** |
| `minRadius` | 25% tinggi RoI | batas bawah radius helm |
| `maxRadius` | 65% tinggi RoI | batas atas radius helm |

Nilai radius mengikuti gagasan pada proposal: **radius ≈ 50% tinggi RoI**, dengan asumsi
ukuran helm tidak melebihi RoI yang telah ditentukan.

### 7.3 Dari Lingkaran ke Sub-window

Lingkaran terbaik (nilai akumulator tertinggi) diubah menjadi **sub-window** berbentuk persegi
yang melingkupi lingkaran tersebut. Sub-window inilah yang menjadi input tahap HOG.

> **Keterbatasan yang diakui.** Sesuai proposal §2.3, CHT **tidak dapat** membedakan helm dari
> kepala tanpa helm karena keduanya berbentuk lingkaran. CHT hanya berperan sebagai
> **pelokalisasi**, bukan penentu keputusan. Keputusan akhir dilakukan oleh HOG + SVM.

In [ ]:
CHT_PARAMS = dict(dp=1.2, param1=100, param2=18,
                  r_min=0.25, r_max=0.65, blur=5)

def cht_detect(gray_roi, params=CHT_PARAMS):
    """Deteksi satu lingkaran terbaik pada RoI. Kembalikan (cx, cy, r) atau None."""
    g = cv2.medianBlur(gray_roi, params["blur"]) if params["blur"] else gray_roi
    h, w = g.shape[:2]
    if h < 20 or w < 20:
        return None
    circles = cv2.HoughCircles(
        g, cv2.HOUGH_GRADIENT,
        dp=params["dp"],
        minDist=max(10, h // 2),
        param1=params["param1"],
        param2=params["param2"],
        minRadius=max(4, int(params["r_min"] * h)),
        maxRadius=max(6, int(params["r_max"] * h)),
    )
    if circles is None:
        return None
    cx, cy, r = circles[0][0]
    return float(cx), float(cy), float(r)


def cht_subwindow_norm(img, rider_box, params=CHT_PARAMS,
                       top=HEAD_BAND_TOP, bottom=HEAD_BAND_BOTTOM):
    """Jalankan CHT pada RoI kepala -> sub-window ternormalisasi (x1,y1,x2,y2).

    Mengembalikan None jika tidak ada lingkaran terdeteksi.
    """
    H, W = img.shape[:2]
    X1, Y1, X2, Y2 = rider_roi_px(img.shape, rider_box, top, bottom)
    roi = crop_px(img, (X1, Y1, X2, Y2))
    if roi is None:
        return None
    gray = to_grayscale(roi)
    res = cht_detect(gray, params)
    if res is None:
        return None
    cx, cy, r = res
    return ((X1 + cx - r) / W, (Y1 + cy - r) / H,
            (X1 + cx + r) / W, (Y1 + cy + r) / H)


def head_band_norm(rider_box, top=HEAD_BAND_TOP, bottom=HEAD_BAND_BOTTOM, pad=0.15):
    """Fallback / baseline: sub-window = pita kepala tanpa CHT (dengan padding)."""
    x1, y1, x2, y2 = rider_box
    w, h = x2 - x1, y2 - y1
    return (x1 + 0.15 * w - pad * w,
            y1 + top * h - pad * h,
            x2 - 0.15 * w + pad * w,
            y1 + bottom * h + pad * h)

print("Parameter CHT:", CHT_PARAMS)

In [ ]:
# Demonstrasi CHT pada beberapa pengendara
demo_rows = df[df["n_rider"] > 0].head(4)
fig, axes = plt.subplots(len(demo_rows), 4, figsize=(12, 3.0 * len(demo_rows)))

for i, (_, row) in enumerate(demo_rows.iterrows()):
    img = imread_robust(row["image_path"])
    H, W = img.shape[:2]
    rider = [b for b in row["boxes"] if b["cls"] == 2][0]
    rb = box_tuple(rider)

    X1, Y1, X2, Y2 = rider_roi_px(img.shape, rb)
    roi = crop_px(img, (X1, Y1, X2, Y2))
    gray = to_grayscale(roi)
    res = cht_detect(gray)

    axes[i, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[i, 0].add_patch(Rectangle((X1, Y1), X2-X1, Y2-Y1, fill=False, ec="crimson", lw=1.8))
    axes[i, 0].set_title("RoI kepala", fontsize=8)

    axes[i, 1].imshow(gray, cmap="gray")
    axes[i, 1].set_title("Grayscale + median blur", fontsize=8)

    if res is not None:
        cx, cy, r = res
        vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
        cv2.circle(vis, (int(cx), int(cy)), int(r), (255, 80, 80), 2)
        cv2.circle(vis, (int(cx), int(cy)), 2, (255, 255, 0), -1)
        axes[i, 2].imshow(vis)
        axes[i, 2].set_title(f"CHT: r={r:.0f}px", fontsize=8)

        sw = cht_subwindow_norm(img, rb)
        crop = crop_px(img, (int(sw[0]*W), int(sw[1]*H), int(sw[2]*W), int(sw[3]*H)))
        if crop is not None:
            axes[i, 3].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        axes[i, 3].set_title("Sub-window", fontsize=8)
    else:
        axes[i, 2].text(0.5, 0.5, "tidak ada\nlingkaran", ha="center", va="center",
                        transform=axes[i, 2].transAxes, fontsize=9, color="crimson")
        axes[i, 2].set_title("CHT: gagal", fontsize=8)
        axes[i, 3].axis("off")

    for ax in axes[i][:3]:
        ax.axis("off")
    if res is not None:
        axes[i, 3].axis("off")

plt.tight_layout()
plt.show()

### 7.4 Mengukur Kinerja Tahap CHT

Sebelum menilai klasifikasi, penting mengukur **apakah CHT berhasil melokalisasi area kepala**.
Metrik yang digunakan: proporsi pengendara yang sub-window hasil CHT-nya **tumpang tindih**
dengan *bounding box* helm *ground truth* (IoU > 0.15).

def cht_localization_rate(df_sub):
    hit = miss = none = 0
    for _, row in df_sub.iterrows():
        img = imread_robust(row["image_path"])
        if img is None:
            continue
        riders = [b for b in row["boxes"] if b["cls"] == 2]
        helmets = [b for b in row["boxes"] if b["cls"] in (0, 1)]
        for r in riders:
            rb = box_tuple(r)
            gts = [box_tuple(h) for h in helmets if iou_norm(rb, box_tuple(h)) > 0.001]
            if not gts:
                continue
            sw = cht_subwindow_norm(img, rb)
            if sw is None:
                none += 1
                continue
            if any(iou_norm(sw, g) > 0.15 for g in gts):
                hit += 1
            else:
                miss += 1
    total = hit + miss + none
    return {"hit": hit, "miss": miss, "no_circle": none,
            "rate": hit / total if total else 0.0, "total": total}

cht_stats = cht_localization_rate(df)
print("Evaluasi pelokalisasian CHT (seluruh dataset)")
print(f"  total pengendara      : {cht_stats['total']}")
print(f"  berhasil dilokalisasi : {cht_stats['hit']}")
print(f"  meleset               : {cht_stats['miss']}")
print(f"  tidak ada lingkaran   : {cht_stats['no_circle']}")
print(f"  localization rate     : {cht_stats['rate']:.1%}")